# Member Conversion Prediction Model
## Week 1 — Problem Definition & Dataset Preparation

**Objective:** predict whether a free-tier learner converts to a paid subscription using information available early in their journey.

> **Data note:** This project uses a public 365 Data Science e-learning dataset as a proxy because IAASSE historical membership data was not available. This is not IAASSE member data, and results must not be described as IAASSE business results.

### Week 1 scope
- Define the prediction target and prediction window.
- Document the dataset and its limitations.
- Load the prepared student-level CSV.
- Run basic integrity checks.
- Record open questions before moving to EDA.

**Not in scope yet:** model training, tuning, model selection, or a prediction interface.

## 1. Problem Definition

This is a **binary classification** problem.

- **Unit of analysis:** one student.
- **Observation window:** first 7 days after registration.
- **Prediction window:** days 8–97 after registration.
- **Target:** `converted_to_paid`
  - `0` = no first paid purchase in the prediction window.
  - `1` = first paid purchase in the prediction window.

Only behavior recorded during the observation window should be used as predictors. Purchase dates and activity after the observation window must not be predictors because they can reveal the outcome or future information.

## 2. Dataset Source & Provenance

The source data comes from the 365 Data Science Learning Data Challenge. Several source tables were aggregated into one student-level table.

Source tables used:
- student information
- student engagement
- student learning
- student exams
- student hub questions
- course ratings
- student purchases (**used to derive the target, not as model features**)

The prepared file contains aggregated features from the first 7 days after registration and a conversion label for the following 90 days.

**Limitations to document**
1. This is a proxy dataset, not IAASSE data.
2. The conversion class may be rare, so accuracy alone will be misleading.
3. The target depends on the chosen 7-day observation and 90-day prediction windows.
4. Check the source dataset's terms before publishing the CSV publicly.

## 3. Load the Prepared Dataset

Place the CSV in your repository at `data/IAASSE_member_conversion_ml_ready.csv`. If your file is stored elsewhere, change `DATA_PATH` below.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../data/IAASSE_member_conversion_ml_ready.csv")
df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())


## 4. Basic Structure

Check column names, data types, and whether each row represents a unique student.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nDuplicate rows:", df.duplicated().sum())
print("Duplicate student IDs:", df["student_id"].duplicated().sum())


## 5. Validate the Target

Confirm that the target exists, is binary, and has both classes. Record class balance now.

In [ ]:
TARGET = "converted_to_paid"

assert TARGET in df.columns, f"Missing target column: {TARGET}"
assert set(df[TARGET].dropna().unique()).issubset({0, 1}), "Target must contain only 0 and 1."
assert df[TARGET].notna().all(), "Target contains missing values."

target_counts = df[TARGET].value_counts().sort_index()
target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": (target_counts / len(df) * 100).round(2)
})
display(target_summary.rename(index={0: "No conversion (0)", 1: "Conversion (1)"}))
print(f"Overall conversion rate: {df[TARGET].mean():.2%}")


## 6. Check Missing Values & Numeric Ranges

This is an initial quality check, not full EDA. Do not automatically drop rows or transform variables just because this cell reveals an issue; investigate first.

In [ ]:
missing = df.isna().sum().rename("missing_count").to_frame()
missing["missing_percentage"] = (missing["missing_count"] / len(df) * 100).round(2)
display(missing.sort_values("missing_percentage", ascending=False))

display(df.describe(include="all").T)


## 7. Temporal & Leakage Review

The prepared table should contain only predictors based on information available in the first 7 days. The purchase table was used to construct the target; its raw purchase fields should not be predictors.

`date_registered` is retained for auditability and temporal validation. Convert it to datetime and do not pass it as a raw string to a model.

In [ ]:
df["date_registered"] = pd.to_datetime(df["date_registered"], errors="coerce")
print("Unparseable registration dates:", df["date_registered"].isna().sum())

forbidden_feature_names = {
    "purchase_id", "purchase_type", "date_purchased",
    "first_purchase", "days_to_purchase"
}
unexpected = forbidden_feature_names.intersection(df.columns)
print("Purchase/outcome fields found in prepared CSV:", unexpected)
assert not unexpected, "Review the prepared dataset: purchase/outcome fields are present."


## 8. Feature Groups (Documentation Only)

This is a working inventory. Preprocessing and feature selection will be handled during EDA/model preparation.

- **Student/context:** `student_country`, `registration_month`, `registration_weekday`
- **Engagement:** engagement records, quizzes/exams/lessons, active days
- **Learning:** learning records, minutes watched, unique courses, active days
- **Exam activity:** attempts, results, completion time, active days
- **Community:** hub questions and active days
- **Ratings:** rating count and average rating

`student_id` is an identifier, not a predictor. `date_registered` is retained for traceability and may help create temporal splits, but should not be used as a raw model input.

## 9. Week 1 Summary

Complete this after running the checks.

- **Problem defined:** predict paid conversion during days 8–97 using activity from days 0–7.
- **Dataset source:** 365 Data Science public e-learning dataset, used as a proxy rather than IAASSE data.
- **Prepared dataset:** one row per student, aggregated early activity features, binary target.
- **Checks completed:** row/column counts, duplicate IDs, target balance, missing values, and obvious leakage fields.
- **Known limitation:** conversion is rare; later evaluation should consider precision, recall, F1, and the confusion matrix rather than accuracy alone.

### Questions to resolve
1. Does IAASSE approve using an external proxy dataset?
2. Are the source dataset's terms compatible with the intended repository/publication?
3. Should the project keep the 7-day observation / 90-day prediction window, or does IAASSE provide a more specific definition?

### Week 1 Check-in notes
Write your own short check-in in IAASSE, summarizing what you completed and what remains uncertain.